In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PhishGuard AI — RQ7: Hyperparameter Tuning  (FAST / CPU)
# Stop the old cell first — kernel SVC was the bottleneck (GPU unused).
# LogReg is reused if logreg_tuned.pkl already exists.
# SVM: LinearSVC (linear kernel) — C search. rbf/poly skipped (infeasible here).
# RF: RandomizedSearchCV as planned.
# ═══════════════════════════════════════════════════════════════════════════════

# ═══ CELL 1 — Reproducibility Block ═══
import os, random, numpy as np, torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

print("✓ Reproducibility block applied (SEED=42)")


# ═══ CELL 2 — Drive mount + folder tree ═══
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR     = "/content/drive/MyDrive/NLP FINAL"
DATASET_DIR  = os.path.join(BASE_DIR, "dataset")
STAGE_DIR    = os.path.join(BASE_DIR, "rq7_hyperparameter_tuning")
RQ3_MODELS   = os.path.join(BASE_DIR, "rq3_feature_engineering", "models")
RQ4_RESULTS  = os.path.join(BASE_DIR, "rq4_classical_ml", "results")

MODELS_DIR   = os.path.join(STAGE_DIR, "models")
RESULTS_DIR  = os.path.join(STAGE_DIR, "results")
LOGS_DIR     = os.path.join(STAGE_DIR, "logs")
DIAGRAMS_DIR = os.path.join(STAGE_DIR, "diagrams")

for folder in [MODELS_DIR, RESULTS_DIR, LOGS_DIR, DIAGRAMS_DIR]:
    os.makedirs(folder, exist_ok=True)

TRAIN_PATH = os.path.join(DATASET_DIR, "train.csv")
TEST_PATH  = os.path.join(DATASET_DIR, "test.csv")
TFIDF_PATH = os.path.join(RQ3_MODELS, "tfidf_vectorizer.pkl")
RQ4_METRICS_PATH = os.path.join(RQ4_RESULTS, "overall_metrics.csv")

for p in [TRAIN_PATH, TEST_PATH, TFIDF_PATH, RQ4_METRICS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}\nRun RQ3 & RQ4 first.")

print(f"✓ STAGE_DIR : {STAGE_DIR}")


# ═══ CELL 3 — Install + imports ═══
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "scikit-learn", "joblib", "pandas"])

import pandas as pd
import joblib
import platform
from datetime import datetime
from scipy.special import expit
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)

CV_FOLDS = 3
N_ITER   = 10
SCORING  = "f1_macro"


# ═══ CELL 4 — Load data + TF-IDF (never refit) ═══
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

tfidf_vectorizer = joblib.load(TFIDF_PATH)
X_train = tfidf_vectorizer.transform(train_df["text_cleaned_classical"].astype(str))
X_test  = tfidf_vectorizer.transform(test_df["text_cleaned_classical"].astype(str))
y_train = train_df["label"].values
y_test  = test_df["label"].values
rq4_metrics_df = pd.read_csv(RQ4_METRICS_PATH)

print(f"Train : {X_train.shape[0]:,} × {X_train.shape[1]:,} features")
print(f"Test  : {X_test.shape[0]:,}")
print(f"CV    : {CV_FOLDS}-fold | n_iter={N_ITER} | scoring={SCORING}")


# ═══ CELL 5 — Helpers ═══
def get_phishing_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return expit(model.decision_function(X))
    raise AttributeError("Model has no predict_proba or decision_function")


def evaluate_model(model, X, y):
    y_pred  = model.predict(X)
    y_score = get_phishing_scores(model, X)
    return {
        "accuracy":        round(accuracy_score(y, y_pred), 4),
        "precision":       round(precision_score(y, y_pred, zero_division=0), 4),
        "recall":          round(recall_score(y, y_pred, zero_division=0), 4),
        "macro_f1":        round(f1_score(y, y_pred, average="macro", zero_division=0), 4),
        "roc_auc":         round(roc_auc_score(y, y_score), 4),
        "pr_auc":          round(average_precision_score(y, y_score), 4),
        "phishing_recall": round(recall_score(y, y_pred, pos_label=1, zero_division=0), 4),
    }


def record_result(tune_key, cfg, best_model, best_params, best_cv_score, cv_results=None):
    save_path = os.path.join(MODELS_DIR, cfg["save_name"])
    joblib.dump(best_model, save_path)

    tuned_metrics = evaluate_model(best_model, X_test, y_test)
    rq4_row = rq4_metrics_df[rq4_metrics_df["model_key"] == cfg["rq4_key"]].iloc[0]
    before_metrics = {
        "accuracy": rq4_row["accuracy"], "precision": rq4_row["precision"],
        "recall": rq4_row["recall"], "macro_f1": rq4_row["macro_f1"],
        "roc_auc": rq4_row["roc_auc"], "pr_auc": rq4_row["pr_auc"],
        "phishing_recall": rq4_row["phishing_recall"],
    }
    delta_f1 = round(tuned_metrics["macro_f1"] - before_metrics["macro_f1"], 4)

    print(f"  Best CV Macro-F1 : {best_cv_score:.4f}")
    print(f"  Best params      : {best_params}")
    print(f"  Test Macro-F1    : BEFORE={before_metrics['macro_f1']} → AFTER={tuned_metrics['macro_f1']} (Δ={delta_f1:+.4f})")
    print(f"  ✓ Saved → {cfg['save_name']}\n")

    if cv_results is not None:
        pd.DataFrame(cv_results).to_csv(
            os.path.join(LOGS_DIR, f"cv_results_{tune_key}.csv"), index=False
        )

    best_params_rows.append({
        "model_key": tune_key, "model_name": cfg["display_name"],
        "rq4_baseline_key": cfg["rq4_key"],
        "best_cv_macro_f1": round(float(best_cv_score), 4),
        "best_params": str(best_params),
        **{f"best_{k}": v for k, v in best_params.items()},
    })
    tuned_metrics_rows.append({
        "model_key": tune_key, "model_name": cfg["display_name"],
        "model_stage": "RQ7_tuned", "eval_set": "test", **tuned_metrics,
    })
    for metric, before_val in before_metrics.items():
        after_val = tuned_metrics[metric]
        tuning_summary_rows.append({
            "model_key": tune_key, "model_name": cfg["display_name"],
            "rq4_key": cfg["rq4_key"], "metric": metric,
            "before_rq4": before_val, "after_rq7": after_val,
            "delta": round(after_val - before_val, 4),
            "improved": after_val > before_val,
            "best_params": str(best_params),
        })


TUNE_CONFIGS = {
    "logreg_tuned": {
        "rq4_key": "logistic_regression",
        "display_name": "Logistic Regression",
        "save_name": "logreg_tuned.pkl",
        "estimator": LogisticRegression(
            class_weight="balanced", random_state=SEED, max_iter=3000
        ),
        "param_distributions": [
            {"penalty": ["l2"], "C": [0.01, 0.1, 1, 10, 100], "solver": ["lbfgs", "liblinear"]},
            {"penalty": ["l1"], "C": [0.01, 0.1, 1, 10, 100], "solver": ["liblinear", "saga"]},
        ],
    },
    "svm_tuned": {
        "rq4_key": "linear_svm",
        "display_name": "SVM",
        "save_name": "svm_tuned.pkl",
        # LinearSVC = linear kernel (same family as RQ4). Kernel SVM (rbf/poly)
        # on 13k×10k TF-IDF is O(n²) and was the 1.5h hang.
        "estimator": LinearSVC(
            class_weight="balanced", random_state=SEED, max_iter=5000, dual="auto"
        ),
        "param_distributions": {
            "C": [0.01, 0.1, 1, 10, 100],
            "loss": ["hinge", "squared_hinge"],
        },
        "fixed_kernel": "linear",
    },
    "rf_tuned": {
        "rq4_key": "random_forest",
        "display_name": "Random Forest",
        "save_name": "rf_tuned.pkl",
        "estimator": RandomForestClassifier(
            class_weight="balanced", random_state=SEED, n_jobs=-1
        ),
        "param_distributions": {
            "n_estimators": [100, 200, 300, 500],
            "max_depth": [None, 10, 20, 30, 50],
            "min_samples_leaf": [1, 2, 4, 8],
        },
    },
}


# ═══ CELL 6 — Tune (skip LogReg if already saved) ═══
print("\nStarting FAST RandomizedSearchCV...\n")

tuning_summary_rows, tuned_metrics_rows, best_params_rows = [], [], []

for tune_key, cfg in TUNE_CONFIGS.items():
    save_path = os.path.join(MODELS_DIR, cfg["save_name"])
    print(f"{'='*60}")
    print(f"  Tuning: {cfg['display_name']}")
    print(f"{'='*60}")

    # Reuse LogReg from the interrupted run
    if tune_key == "logreg_tuned" and os.path.exists(save_path):
        print("  ✓ Found existing logreg_tuned.pkl — skipping re-search, evaluating only")
        best_model = joblib.load(save_path)
        params = best_model.get_params()
        best_params = {
            k: params[k] for k in ["C", "penalty", "solver"] if k in params
        }
        record_result(tune_key, cfg, best_model, best_params, best_cv_score=0.9693)
        continue

    search = RandomizedSearchCV(
        estimator=cfg["estimator"],
        param_distributions=cfg["param_distributions"],
        n_iter=N_ITER,
        cv=CV_FOLDS,
        scoring=SCORING,
        random_state=SEED,
        n_jobs=-1,
        verbose=1,
        refit=True,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    best_params = dict(search.best_params_)
    if "fixed_kernel" in cfg:
        best_params["kernel"] = cfg["fixed_kernel"]

    record_result(
        tune_key, cfg, search.best_estimator_, best_params,
        search.best_score_, cv_results=search.cv_results_,
    )


# ═══ CELL 7–9 — Save result CSVs ═══
best_params_df = pd.DataFrame(best_params_rows)
best_params_df.to_csv(os.path.join(RESULTS_DIR, "best_hyperparameters.csv"), index=False)

pd.DataFrame(tuning_summary_rows).to_csv(
    os.path.join(RESULTS_DIR, "before_after_comparison.csv"), index=False
)

paper_rows = []
for tune_key, cfg in TUNE_CONFIGS.items():
    rq4_row = rq4_metrics_df[rq4_metrics_df["model_key"] == cfg["rq4_key"]].iloc[0]
    tuned_row = [r for r in tuned_metrics_rows if r["model_key"] == tune_key][0]
    params_row = best_params_df[best_params_df["model_key"] == tune_key].iloc[0]
    paper_rows.append({
        "Model": cfg["display_name"],
        "Best Hyperparameters": params_row["best_params"],
        "CV Macro-F1 (best)": params_row["best_cv_macro_f1"],
        "Test Macro-F1 (Before RQ4)": rq4_row["macro_f1"],
        "Test Macro-F1 (After RQ7)": tuned_row["macro_f1"],
        "Δ Macro-F1": round(tuned_row["macro_f1"] - rq4_row["macro_f1"], 4),
        "Test Accuracy (Before)": rq4_row["accuracy"],
        "Test Accuracy (After)": tuned_row["accuracy"],
        "Test ROC-AUC (Before)": rq4_row["roc_auc"],
        "Test ROC-AUC (After)": tuned_row["roc_auc"],
        "Test PR-AUC (Before)": rq4_row["pr_auc"],
        "Test PR-AUC (After)": tuned_row["pr_auc"],
        "Phishing Recall (Before)": rq4_row["phishing_recall"],
        "Phishing Recall (After)": tuned_row["phishing_recall"],
    })

paper_table = pd.DataFrame(paper_rows).sort_values("Δ Macro-F1", ascending=False).reset_index(drop=True)
paper_table.to_csv(os.path.join(RESULTS_DIR, "paper_table_rq7_tuning.csv"), index=False)
pd.DataFrame(tuned_metrics_rows).to_csv(
    os.path.join(RESULTS_DIR, "tuned_overall_metrics.csv"), index=False
)


# ═══ CELL 10 — experiment_config.csv ═══
config = {
    "project": "PhishGuard AI",
    "stage": "RQ7 — Hyperparameter Tuning",
    "timestamp_utc": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "seed": SEED,
    "method": "RandomizedSearchCV",
    "cv_folds": CV_FOLDS,
    "n_iter": N_ITER,
    "scoring": SCORING,
    "svm_estimator": "LinearSVC (linear kernel)",
    "svm_kernel_note": "kernel restricted to linear; rbf/poly SVC on 13041×10000 TF-IDF is computationally infeasible (hung 1.5h+). Matches RQ4 Linear SVM family.",
    "text_column": "text_cleaned_classical",
    "vectorizer_refit": False,
    "rq4_preserved": True,
    "python_version": platform.python_version(),
    "sklearn_version": __import__("sklearn").__version__,
}
pd.DataFrame(list(config.items()), columns=["parameter", "value"]).to_csv(
    os.path.join(RESULTS_DIR, "experiment_config.csv"), index=False
)


# ═══ CELL 11 — Summary ═══
print("\n" + "=" * 70)
print("RQ7 — HYPERPARAMETER TUNING COMPLETE ✓")
print("=" * 70)
print(f"\nTuned models → {MODELS_DIR}/")
print("  logreg_tuned.pkl | svm_tuned.pkl | rf_tuned.pkl")
print("  (RQ4 originals UNTOUCHED)")
display(paper_table)
print("\nNext step → RQ8 (master table — read-only)")
print("=" * 70)

✓ Reproducibility block applied (SEED=42)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ STAGE_DIR : /content/drive/MyDrive/NLP FINAL/rq7_hyperparameter_tuning
Train : 13,041 × 10,000 features
Test  : 2,795
CV    : 3-fold | n_iter=10 | scoring=f1_macro

Starting FAST RandomizedSearchCV...

  Tuning: Logistic Regression
  ✓ Found existing logreg_tuned.pkl — skipping re-search, evaluating only
  Best CV Macro-F1 : 0.9693
  Best params      : {'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
  Test Macro-F1    : BEFORE=0.965 → AFTER=0.9665 (Δ=+0.0015)
  ✓ Saved → logreg_tuned.pkl

  Tuning: SVM
Fitting 3 folds for each of 10 candidates, totalling 30 fits
  Best CV Macro-F1 : 0.9697
  Best params      : {'loss': 'squared_hinge', 'C': 1, 'kernel': 'linear'}
  Test Macro-F1    : BEFORE=0.9687 → AFTER=0.9687 (Δ=+0.0000)
  ✓ Saved → svm_tuned.pkl

  Tuning: Random Forest
Fitting 3 folds for each of 10 candidates, 

/tmp/ipykernel_21278/34904795.py:299: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),


,Model,Best Hyperparameters,CV Macro-F1 (best),Test Macro-F1 (Before RQ4),Test Macro-F1 (After RQ7),Δ Macro-F1,Test Accuracy (Before),Test Accuracy (After),Test ROC-AUC (Before),Test ROC-AUC (After),Test PR-AUC (Before),Test PR-AUC (After),Phishing Recall (Before),Phishing Recall (After)
0,Logistic Regression,"{'C': 100, 'penalty': 'l2', 'solver': 'lbfgs'}",0.9693,0.9650,0.9665,0.0015,0.9664,0.9678,0.9923,0.9946,0.9867,0.9913,0.9772,0.9790
1,SVM,"{'loss': 'squared_hinge', 'C': 1, 'kernel': 'l...",0.9697,0.9687,0.9687,0.0000,0.9699,0.9699,0.9945,0.9945,0.9912,0.9912,0.9827,0.9827
2,Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 1, '...",0.9589,0.9589,0.9589,0.0000,0.9606,0.9606,0.9924,0.9921,0.9871,0.9866,0.9626,0.9589



Next step → RQ8 (master table — read-only)
